# 3.2절 데이터 EDA 그림·표 생성

- H&M → Dunnhumby 순서
- 최종 분할: 마지막 1주 test, 그 앞 전체 학습
- 출력: `내 드라이브/논문/figures/ch3_eda/`에 `fig_hm_eda.png`(3×2, 재구매 패널 포함), `fig_dh_eda.png`(3×2), `fig_compare_eda.png`, `table_data_stats.csv`
- 런타임: CPU로 충분합니다. H&M 3,100만 행을 읽으므로 **고용량 RAM** 런타임을 권장합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!apt-get -qq install -y fonts-nanum > /dev/null
import os
for p in ['/content/drive/MyDrive/논문/data/raw/hm/transactions_train.csv','/content/drive/MyDrive/논문/data/raw/hm/articles.csv',
          '/content/drive/MyDrive/논문/data/raw/dunnhumby/transaction_data.csv','/content/drive/MyDrive/논문/data/raw/dunnhumby/product.csv']:
    print('있음' if os.path.exists(p) else '없음 → 경로 확인', p)

In [ ]:
# 3.2절 데이터 EDA 그림·표 생성 (H&M → Dunnhumby 순)
# Colab: 아래 BASE만 Drive 경로로 두고 그대로 실행. 로컬: 환경변수 EDA_LOCAL=1.
import os, numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
from matplotlib.ticker import FuncFormatter
import matplotlib.dates as mdates

if os.environ.get("EDA_LOCAL"):
    HM_DIR = "/Users/jungun/Workspace/논문준비/data/hm"
    DH_DIR = "/Users/jungun/Workspace/논문준비/data/dunnhumby/dunnhumby_The-Complete-Journey CSV"
    OUT = "/Users/jungun/Workspace/논문준비/1.최종문서/그림/eda_out"
else:
    BASE = "/content/drive/MyDrive/논문/data/raw"
    HM_DIR, DH_DIR = f"{BASE}/hm", f"{BASE}/dunnhumby"
    OUT = "/content/drive/MyDrive/논문/figures/ch3_eda"
os.makedirs(OUT, exist_ok=True)

# 최종 평가 분할: 마지막 1주 test, 그 앞 전체 학습 (AGENTS.md P6)
HM_TEST_START, DH_TEST_START = pd.Timestamp("2020-09-16"), 705

# ── 한글 글꼴 ────────────────────────────────────────────────
cands = [f for f in fm.findSystemFonts() if any(k in f for k in ("NanumGothic", "AppleSDGothicNeo"))]
assert cands, "한글 글꼴이 없습니다. Colab이면 첫 셀의 apt-get install fonts-nanum 을 실행하세요."
fm.fontManager.addfont(cands[0])
plt.rcParams["font.family"] = [fm.FontProperties(fname=cands[0]).get_name(), "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["axes.formatter.use_mathtext"] = False
plt.rcParams["mathtext.fontset"] = "dejavusans"
def logfmt(v, _):
    e = np.log10(v); return f"10$^{{{int(round(e))}}}$" if abs(e - round(e)) < 1e-9 else ""
C_HM, C_DH = "#8c8c8c", "#1f1f1f"

# ── 데이터 읽기 ──────────────────────────────────────────────
hm_raw = pd.read_csv(f"{HM_DIR}/transactions_train.csv",
                     dtype={"customer_id": "string", "article_id": "int64", "price": "float64", "sales_channel_id": "int8"})
hm_dups = int(hm_raw.duplicated().sum()); hm_rows_raw = len(hm_raw)
hm = hm_raw.drop_duplicates(); del hm_raw
hm["t"] = pd.to_datetime(hm["t_dat"])
art = pd.read_csv(f"{HM_DIR}/articles.csv", usecols=["article_id", "product_group_name", "product_type_name"])
dh = pd.read_csv(f"{DH_DIR}/transaction_data.csv",
                 usecols=["household_key", "BASKET_ID", "DAY", "PRODUCT_ID", "QUANTITY", "SALES_VALUE", "STORE_ID"])
prod = pd.read_csv(f"{DH_DIR}/product.csv", usecols=["PRODUCT_ID", "DEPARTMENT", "COMMODITY_DESC", "SUB_COMMODITY_DESC"])

# 장바구니 단위: H&M = 고객×구매일, Dunnhumby = BASKET_ID
hm_b = hm.groupby(["customer_id", "t"], sort=False).agg(n=("article_id", "size"), v=("price", "sum")).reset_index()
dh_b = dh.groupby("BASKET_ID", sort=False).agg(u=("household_key", "first"), day=("DAY", "first"),
                                              n=("PRODUCT_ID", "size"), v=("SALES_VALUE", "sum")).reset_index()

def stats(name, tx, u, i, b_u, b_t, basket, t_col, is_date):
    s = {}
    nb = basket.groupby(b_u).size()
    span = tx.groupby(u)[t_col].agg(["min", "max"])
    span = (span["max"] - span["min"]).dt.days if is_date else span["max"] - span["min"]
    bt = basket[[b_u, b_t]].drop_duplicates().sort_values([b_u, b_t])
    gap = bt.groupby(b_u)[b_t].diff().dropna()
    gap = gap.dt.days if is_date else gap
    pair = tx.groupby([u, i]).size() if name == "H&M" else None
    rep = (tx.drop_duplicates([u, i, t_col]).groupby([u, i]).size() if name == "H&M"
           else tx.groupby([u, i])["BASKET_ID"].nunique())
    pop = tx.groupby(i)[u].nunique().sort_values(ascending=False).values
    nu, ni = tx[u].nunique(), tx[i].nunique()
    s.update({"고객 수": nu, "상품 수": ni, "거래 행 수": len(tx),
              "장바구니당 상품 수(중앙값)": basket["n"].median(),
              "고객당 장바구니 수(중앙값)": nb.median(),
              "구매일 간격(중앙값, 일)": gap.median(),
              "첫~마지막 구매 기간(중앙값, 일)": span.median(),
              "고객당 구매 상품 수(중앙값)": tx.groupby(u)[i].nunique().median(),
              "장바구니 1개뿐인 고객 비율(%)": 100 * (nb == 1).mean(),
              "반복 구매 쌍 비율(%)": 100 * (rep > 1).mean(),
              "상위 10% 상품의 구매 쌍 점유율(%)": 100 * pop[: len(pop) // 10].sum() / pop.sum(),
              "한 고객만 산 상품 비율(%)": 100 * (pop == 1).mean(),
              "고객-상품 행렬 밀도(%)": 100 * len(rep) / (nu * ni)})
    return s, nb.values, gap.values, pop, rep.values

hm_s, hm_nb, hm_gap, hm_pop, hm_rep = stats("H&M", hm, "customer_id", "article_id", "customer_id", "t", hm_b, "t", True)
dh_s, dh_nb, dh_gap, dh_pop, dh_rep = stats("Dunnhumby", dh, "household_key", "PRODUCT_ID", "u", "day", dh_b, "DAY", False)

# 학습/test 분할 규모 (신규 상품 정답: 학습에 있던 고객·상품, 학습에서 산 적 없는 쌍)
def split_info(tx, u, i, is_test):
    tr, te = tx[~is_test], tx[is_test]
    te = te[te[u].isin(set(tr[u])) & te[i].isin(set(tr[i]))]
    trp = pd.MultiIndex.from_frame(tr[[u, i]].drop_duplicates())
    tep = pd.MultiIndex.from_frame(te[[u, i]].drop_duplicates())
    new = tep.difference(trp)
    # ③ test 주 구매(학습에 있던 고객의 고유 고객–상품 쌍) 가운데 학습 기간에 이미 산 상품의 비율
    tu = tx[is_test]; tu = tu[tu[u].isin(set(tr[u]))]
    allp = pd.MultiIndex.from_frame(tu[[u, i]].drop_duplicates())
    rep_share = 100 * allp.isin(trp).mean()
    return {"학습 행 수": len(tr), "test 평가 고객": new.get_level_values(0).nunique(), "test 신규 정답 쌍": len(new),
            "test 주 구매 중 재구매 비율(%)": rep_share}
# ② 구매(장바구니 안 고유 상품) 가운데 그 고객이 이전 장바구니에서 산 적 있는 상품을 다시 산 비율
def repeat_purchase(tx, u, i, b, order):
    x = tx.drop_duplicates([u, i, b]).sort_values(order)
    x = x.assign(rep=x.duplicated([u, i]))
    per = x.groupby(u)["rep"].mean() * 100
    return 100 * x["rep"].mean(), per
hm_rp, hm_per = repeat_purchase(hm, "customer_id", "article_id", "t", ["t"])
dh_rp, dh_per = repeat_purchase(dh, "household_key", "PRODUCT_ID", "BASKET_ID", ["DAY", "BASKET_ID"])
hm_s.update({"재구매 구매 비율(%)": hm_rp, "고객별 재구매 비율 중앙값(%)": hm_per.median(), "재구매 없는 고객 비율(%)": 100 * (hm_per == 0).mean()})
dh_s.update({"재구매 구매 비율(%)": dh_rp, "고객별 재구매 비율 중앙값(%)": dh_per.median(), "재구매 없는 고객 비율(%)": 100 * (dh_per == 0).mean()})
hm_s.update(split_info(hm, "customer_id", "article_id", hm["t"] >= HM_TEST_START))
dh_s.update(split_info(dh, "household_key", "PRODUCT_ID", dh["DAY"] >= DH_TEST_START))
hm_s.update({"원본 거래 행 수": hm_rows_raw, "완전 중복 행": hm_dups})

table = pd.DataFrame({"H&M": hm_s, "Dunnhumby": dh_s})
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
print(table)
table.to_csv(f"{OUT}/table_data_stats.csv", encoding="utf-8-sig")

# ── 데이터별 그림 (3×2) ──────────────────────────────────────
def dataset_figure(name, fname, weekly, test_start, nb, nb_label, cat_share, cat_label, val, val_label, color,
                   per_rep, test_rep):
    fig, axs = plt.subplots(3, 2, figsize=(10, 10.8), dpi=300)
    ax = axs[0, 0]
    ax.plot(weekly.index, weekly.values / 1e3, color=color, lw=1.2)
    ax.axvspan(test_start, weekly.index.max(), color="#d9d9d9", zorder=0)
    ax.text(test_start, ax.get_ylim()[1] * 0.97, "test(마지막 1주) ", fontsize=8, va="top", ha="right")
    ax.set_title("(a) 주별 거래 건수"); ax.set_ylabel("거래 건수(천 건)")
    if isinstance(weekly.index, pd.DatetimeIndex):
        ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m")); ax.tick_params(axis="x", labelsize=8, rotation=30)
        ax.set_xlabel("주 마지막 날짜")
    else:
        ax.set_xlabel("주 마지막 DAY")
    ax = axs[0, 1]
    bins = np.unique(np.round(np.logspace(0, np.log10(nb.max() + 1), 40))) - 0.5
    ax.hist(nb, bins=bins, color=color, edgecolor="white", lw=0.3)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.xaxis.set_major_formatter(FuncFormatter(logfmt)); ax.yaxis.set_major_formatter(FuncFormatter(logfmt))
    ax.set_title("(b) 고객별 장바구니 수 분포"); ax.set_xlabel(nb_label); ax.set_ylabel("고객 수")
    ax = axs[1, 0]
    top = cat_share.head(10)[::-1]
    ax.barh(top.index, top.values, color=color)
    ax.set_title(f"(c) {cat_label}별 거래 비중(상위 10개)"); ax.set_xlabel("거래 비중(%)")
    ax.tick_params(axis="y", labelsize=8)
    ax = axs[1, 1]
    hi = np.quantile(val, 0.99)
    ax.hist(np.clip(val, 0, hi), bins=60, color=color, edgecolor="white", lw=0.3)
    ax.set_title("(d) " + val_label.split("|")[0]); ax.set_xlabel(val_label.split("|")[1]); ax.set_ylabel("빈도")
    ax = axs[2, 0]
    ax.hist(per_rep, bins=np.linspace(0, 100, 21), weights=np.full(len(per_rep), 100 / len(per_rep)),
            color=color, edgecolor="white", lw=0.3)
    ax.set_title("(e) 고객별 재구매 비율 분포"); ax.set_xlabel("고객의 구매 중 재구매 비율(%)"); ax.set_ylabel("고객 비율(%)")
    ax = axs[2, 1]
    vals = [test_rep, 100 - test_rep]
    bars = ax.bar(["재구매\n(학습 기간에 산 상품)", "신규 상품"], vals, color=[color, "white"], edgecolor="black", width=0.55)
    for bb, v in zip(bars, vals): ax.text(bb.get_x() + bb.get_width() / 2, v + 1.5, f"{v:.1f}%", ha="center", fontsize=9)
    ax.set_ylim(0, 110); ax.set_ylabel("test 주 구매 비율(%)"); ax.set_title("(f) test 주 구매의 구성")
    fig.tight_layout()
    fig.savefig(f"{OUT}/{fname}", facecolor="white"); plt.close(fig)

hm_weekly = hm.set_index("t").resample("W-TUE").size()   # 마지막 주(9/16~9/22, 수~화)가 한 칸이 되도록
hm_cat = hm.merge(art, on="article_id")["product_group_name"].value_counts(normalize=True) * 100
dataset_figure("H&M", "fig_hm_eda.png", hm_weekly, HM_TEST_START, hm_nb, "고객당 구매일(장바구니) 수",
               hm_cat, "상품 그룹", hm.groupby("article_id")["price"].median().values,
               "상품별 가격 분포(상위 1%는 99분위 값으로 표시)|상품 가격(상대 척도)", C_HM,
               hm_per.values, hm_s["test 주 구매 중 재구매 비율(%)"])

dh_last = int(dh["DAY"].max())                     # 711: 마지막 주(705~711)가 한 칸이 되도록 끝에서부터 7일씩 묶음
dh_w = (dh_last - dh["DAY"]) // 7
dh_weekly = dh.groupby(dh_w).size().sort_index(ascending=False); dh_weekly.index = dh_last - dh_weekly.index * 7   # 각 주의 마지막 DAY
dh_cat = dh.merge(prod, on="PRODUCT_ID")["DEPARTMENT"].value_counts(normalize=True) * 100
dataset_figure("Dunnhumby", "fig_dh_eda.png", dh_weekly, DH_TEST_START, dh_nb, "가구당 장바구니 수",
               dh_cat, "부서(department)", dh_b["v"].values,
               "장바구니 금액 분포(상위 1%는 99분위 값으로 표시)|장바구니 판매 금액", C_DH,
               dh_per.values, dh_s["test 주 구매 중 재구매 비율(%)"])

# ── 두 데이터 비교 그림 (2×2) ────────────────────────────────
fig, axs = plt.subplots(2, 2, figsize=(10, 7.2), dpi=300)
def ccdf(ax, x, lab, ls, c):
    x = np.sort(x); ax.plot(x, 1 - np.arange(len(x)) / len(x), ls, color=c, label=lab, lw=1.4)
ax = axs[0, 0]; ccdf(ax, hm_nb, "H&M", "--", C_HM); ccdf(ax, dh_nb, "Dunnhumby", "-", C_DH)
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_title("(a) 고객별 구매 횟수 분포")
ax.set_xlabel("고객당 장바구니(구매일) 수"); ax.set_ylabel("해당 값 이상인 고객 비율"); ax.legend()
ax = axs[0, 1]
for x, lab, ls, c in [(hm_pop, "H&M", "--", C_HM), (dh_pop, "Dunnhumby", "-", C_DH)]:
    ax.plot(np.arange(1, len(x) + 1) / len(x), x, ls, color=c, label=lab, lw=1.4)
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_title("(b) 상품 인기도 분포")
ax.set_xlabel("상품 순위(상위 비율)"); ax.set_ylabel("구매 고객 수"); ax.legend()
for a in axs[0]:
    a.xaxis.set_major_formatter(FuncFormatter(logfmt)); a.yaxis.set_major_formatter(FuncFormatter(logfmt))
ax = axs[1, 0]; bins = np.arange(0, 121, 3)
ax.hist(np.clip(hm_gap, 0, 120), bins=bins, density=True, histtype="step", ls="--", color=C_HM, lw=1.4, label="H&M")
ax.hist(np.clip(dh_gap, 0, 120), bins=bins, density=True, histtype="step", color=C_DH, lw=1.4, label="Dunnhumby")
ax.set_title("(c) 구매 간격 분포"); ax.set_xlabel("연속한 구매일 사이 간격(일, 120일 이상은 120)"); ax.set_ylabel("밀도"); ax.legend()
ax = axs[1, 1]; cats = ["1회", "2회", "3~5회", "6회 이상"]
share = lambda r: np.array([np.mean(r == 1), np.mean(r == 2), np.mean((r >= 3) & (r <= 5)), np.mean(r >= 6)]) * 100
xs, w = np.arange(4), 0.38
ax.bar(xs - w / 2, share(hm_rep), w, color="#bdbdbd", edgecolor="black", label="H&M")
ax.bar(xs + w / 2, share(dh_rep), w, color="#3a3a3a", label="Dunnhumby")
ax.set_xticks(xs); ax.set_xticklabels(cats); ax.set_title("(d) 반복 구매 분포")
ax.set_xlabel("같은 고객–상품 쌍의 구매 횟수"); ax.set_ylabel("고객–상품 쌍 비율(%)"); ax.legend()
fig.tight_layout(); fig.savefig(f"{OUT}/fig_compare_eda.png", facecolor="white"); plt.close(fig)

print("반복 구매 분포(%) H&M", share(hm_rep).round(1), "Dunnhumby", share(dh_rep).round(1))
print("상위 카테고리 H&M", hm_cat.head(5).round(1).to_dict())
print("상위 카테고리 Dunnhumby", dh_cat.head(5).round(1).to_dict())
print("저장 위치:", OUT, sorted(os.listdir(OUT)))


In [ ]:
from IPython.display import Image, display
for f in ['fig_hm_eda.png','fig_dh_eda.png','fig_compare_eda.png']:
    print(f); display(Image(f'{OUT}/{f}', width=900))